In [1]:
import pandas as pd
import sqlalchemy
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path


In [2]:
# connecing to the database
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
print("Connected Successfully")

Connected Successfully


In [3]:
#the tables name available in the database 
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""

tables = pd.read_sql(query, engine)

tables

,table_name
0,product_recommendations
1,model_metrics
2,customers
3,orders
4,order_items
5,products
6,sellers
7,payments
8,reviews
9,customer_features


In [ ]:
# just loading the customers table to check the data
customers = pd.read_sql(
    "SELECT * FROM customers",
    engine
)

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,Sao Bernardo Do Campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,Sao Paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,Mogi Das Cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP


In [8]:
# Loading all the tables into dataframes
# checking weather the data is loaded properly or not by checking the shape of the data
customers = pd.read_sql("SELECT * FROM customers", engine)
orders = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
payments = pd.read_sql("SELECT * FROM payments", engine)
reviews = pd.read_sql("SELECT * FROM reviews", engine)
products = pd.read_sql("SELECT * FROM products", engine)

datasets = {
    "customers": customers,
    "orders": orders,
    "payments": payments,
    "order_items": order_items,
    "products": products,
    "reviews": reviews
}

for name, df in datasets.items():
    print(name, df.shape)

customers (99441, 5)
orders (99441, 8)
payments (103886, 5)
order_items (112650, 7)
products (32951, 9)
reviews (99224, 7)


In [9]:
# Checking for missing values in each table
for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    print("\n", name.upper())

    print(missing.sort_values(ascending=False))


 CUSTOMERS
Series([], dtype: int64)

 ORDERS
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

 PAYMENTS
Series([], dtype: int64)

 ORDER_ITEMS
Series([], dtype: int64)

 PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

 REVIEWS
review_comment_title      87658
review_comment_message    58274
dtype: int64


In [ ]:
# order status distributione
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
# checking total revenue generated by  customer
payments["payment_value"].sum()

np.float64(16008872.120000001)

In [14]:
# average revenue generated by ordre
payments["payment_value"].mean()

np.float64(154.10038041699556)

In [16]:
# category analysis
customers["customer_unique_id"].nunique()

96096

In [17]:
# repeated customer analysis
customer_orders = orders.groupby(
    "customer_id"
).size()

(customer_orders > 1).sum()

np.int64(0)

In [ ]:
# review score distribution
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [19]:
# products category distribution
products["product_category_name"].value_counts().head(10)

product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64

JOINING SOME TABLES TO GET PROPER RESULT 


In [ ]:
# verifying the number of unique customers and orders in the database
query = """
SELECT
    COUNT(DISTINCT customer_id) AS customers,
    COUNT(DISTINCT order_id) AS orders
FROM orders
"""

pd.read_sql(query, engine)

,customers,orders
0,99441,99441


In [ ]:
# retrieving the top 10 customers based on total revenue generated
query = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.payment_value)::numeric,2) AS total_revenue
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN payments p
    ON o.order_id = p.order_id
GROUP BY c.customer_unique_id
ORDER BY total_revenue DESC
LIMIT 10
"""

pd.read_sql(query, engine)

,customer_unique_id,total_orders,total_revenue
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08
1,46450c74a0d8c5ca9395da1daac6c120,3,9553.02
2,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63
3,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88
4,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31
5,459bef486812aa25204be022145caa62,1,6922.21
6,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66
7,4007669dec559734d6f53e029e360987,1,6081.54
8,5d0a2980b292d049061542014e8960bf,1,4809.44
9,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34


In [22]:
# retrieving the total revenue generated by each state
query = """
SELECT
    c.customer_state,
    ROUND(SUM(p.payment_value)::numeric,2) AS revenue
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN payments p
    ON o.order_id = p.order_id
GROUP BY c.customer_state
ORDER BY revenue DESC
"""

pd.read_sql(query, engine)

,customer_state,revenue
0,SP,5998226.96
1,RJ,2144379.69
2,MG,1872257.26
3,RS,890898.54
4,PR,811156.38
5,SC,623086.43
6,BA,616645.82
7,DF,355141.08
8,GO,350092.31
9,ES,325967.55


In [24]:
# retrieving the top 10 product categories based on the number of items sold
query = """
SELECT
    p.product_category_name,
    COUNT(*) AS items_sold
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY items_sold DESC
LIMIT 10
"""

pd.read_sql(query, engine)

,product_category_name,items_sold
0,cama_mesa_banho,11115
1,beleza_saude,9670
2,esporte_lazer,8641
3,moveis_decoracao,8334
4,informatica_acessorios,7827
5,utilidades_domesticas,6964
6,relogios_presentes,5991
7,telefonia,4545
8,ferramentas_jardim,4347
9,automotivo,4235


In [27]:
# retrieving the top 10 sellers based on the number of products sold
query = """
SELECT
    seller_id,
    COUNT(*) AS products_sold
FROM order_items
GROUP BY seller_id
ORDER BY products_sold DESC
LIMIT 10
"""

pd.read_sql(query, engine)

,seller_id,products_sold
0,6560211a19b47992c3666cc44a7e94c0,2033
1,4a3ca9315b744ce9f8e9374361493884,1987
2,1f50f920176fa81dab994f9023523100,1931
3,cc419e0650a3c5ba77189a1882b7556a,1775
4,da8622b14eb17ae2831f4ac5b9dab84a,1551
5,955fee9216a65b617aa5c0531780ce60,1499
6,1025f0e2d44d7041d6cf58b6550e0bfa,1428
7,7c67e1448b00f6e969d365cea6b010ab,1364
8,ea8482cd71df3c1969d7b9473ff13abc,1203
9,7a67c85e85bb2ce8582c35f2203ad736,1171


In [ ]:
# analyzing the relationship between review scores and average order value(REVIEW VS REVENUE)
query = """
SELECT
    r.review_score,
    ROUND(AVG(p.payment_value)::numeric,2) AS avg_order_value
FROM reviews r
JOIN orders o
    ON r.order_id = o.order_id
JOIN payments p
    ON o.order_id = p.order_id
GROUP BY r.review_score
ORDER BY r.review_score
"""

pd.read_sql(query, engine)

,review_score,avg_order_value
0,1,186.39
1,2,163.37
2,3,145.13
3,4,147.98
4,5,149.70
